# Reinforcement Learning with SciPy

In [1]:
from enum import IntEnum


# MDP from the previous lecture.
class State(IntEnum):
    S0 = 0
    W = 1
    M = 2
    WM = 3
    SUCCESS = 4
    ABANDON = 5


class Action(IntEnum):
    ASK_WEATHER = 0
    ASK_MOOD = 1
    ASK_BOTH = 2
    RECOMMEND = 3


TERMINAL_STATES = {State.SUCCESS, State.ABANDON}
NON_TERMINAL_STATES = [s for s in State if s not in TERMINAL_STATES]

AVAILABLE_ACTIONS = {
    State.S0: [Action.ASK_WEATHER, Action.ASK_MOOD, Action.ASK_BOTH],
    State.W: [Action.ASK_MOOD],
    State.M: [Action.ASK_WEATHER],
    State.WM: [Action.RECOMMEND],
    State.SUCCESS: [],
    State.ABANDON: [],
}

TRANSITIONS = {
    (State.S0, Action.ASK_WEATHER): {
        State.W: 0.75,
        State.S0: 0.15,
        State.ABANDON: 0.10,
    },
    (State.S0, Action.ASK_MOOD): {
        State.M: 0.70,
        State.S0: 0.20,
        State.ABANDON: 0.10,
    },
    (State.S0, Action.ASK_BOTH): {
        State.WM: 0.50,
        State.W: 0.20,
        State.M: 0.15,
        State.ABANDON: 0.15,
    },
    (State.W, Action.ASK_MOOD): {
        State.WM: 0.80,
        State.W: 0.10,
        State.ABANDON: 0.10,
    },
    (State.M, Action.ASK_WEATHER): {
        State.WM: 0.80,
        State.M: 0.10,
        State.ABANDON: 0.10,
    },
    (State.WM, Action.RECOMMEND): {
        State.SUCCESS: 0.75,
        State.ABANDON: 0.25,
    },
}


def reward(state, action, next_state):
    """Reward is +10 for reaching SUCCESS, 0 otherwise."""
    return 10.0 if next_state == State.SUCCESS else 0.0


In [ ]:
import numpy as np
from scipy.optimize import linprog

states = list(State)
state_to_idx = {state: i for i, state in enumerate(states)}

n_states = len(states)
gamma = 0.99

In [3]:
c = np.zeros(n_states)

for s in NON_TERMINAL_STATES:
    c[state_to_idx[s]] = 1.0

In [ ]:
A_ub = []
b_ub = []

for s in NON_TERMINAL_STATES:
    for a in AVAILABLE_ACTIONS[s]:
        row = np.zeros(n_states)

        # -V(s)
        row[state_to_idx[s]] = -1.0

        expected_reward = 0.0

        for s_next, prob in TRANSITIONS[(s, a)].items():
            # + gamma * p(s' | s, a) * V(s')
            row[state_to_idx[s_next]] += gamma * prob

            expected_reward += prob * reward(s, a, s_next)

        A_ub.append(row)
        b_ub.append(-expected_reward)

A_ub = np.array(A_ub)
b_ub = np.array(b_ub)

In [5]:
terminal_states = [
    State.SUCCESS,
    State.ABANDON,
]

A_eq = []
b_eq = []

for s in terminal_states:
    row = np.zeros(n_states)
    row[state_to_idx[s]] = 1.0

    A_eq.append(row)
    b_eq.append(0.0)

A_eq = np.array(A_eq)
b_eq = np.array(b_eq)

In [6]:
bounds = [(None, None)] * n_states

In [ ]:
result = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs",
)

if not result.success:
    raise RuntimeError(result.message)

V = {s: result.x[state_to_idx[s]] for s in states}

print("Optimal Value Function:")
for s in states:
    print(f"  V({s.name}) = {V[s]:.4f}")
# Optimal Value Function:
#   V(S0) = 5.9969
#   V(W) = 6.5927
#   V(M) = 6.5927
#   V(WM) = 7.5000
#   V(SUCCESS) = -0.0000
#   V(ABANDON) = -0.0000

Optimal Value Function:
  V(S0) = 5.9969
  V(W) = 6.5927
  V(M) = 6.5927
  V(WM) = 7.5000
  V(SUCCESS) = -0.0000
  V(ABANDON) = -0.0000


In [ ]:
print("\nOptimal Policy:")

for s in NON_TERMINAL_STATES:
    action_values = []

    for a in AVAILABLE_ACTIONS[s]:
        q_value = sum(
            prob * (reward(s, a, s_next) + gamma * V[s_next])
            for s_next, prob in TRANSITIONS[(s, a)].items()
        )

        action_values.append((a, q_value))

    best_action, best_q = max(
        action_values,
        key=lambda item: item[1],
    )

    q_str = ", ".join(f"{a.name}={q:.3f}" for a, q in action_values)

    print(f"  π({s.name:8}) = {best_action.name:12} [Q-values: {q_str}]")

# Optimal Policy:
#   π(S0      ) = ASK_BOTH     [Q-values: ASK_WEATHER=5.786, ASK_MOOD=5.756, ASK_BOTH=5.997]
#   π(W       ) = ASK_MOOD     [Q-values: ASK_MOOD=6.593]
#   π(M       ) = ASK_WEATHER  [Q-values: ASK_WEATHER=6.593]
#   π(WM      ) = RECOMMEND    [Q-values: RECOMMEND=7.500]    


Optimal Policy:
  π(S0      ) = ASK_BOTH     [Q-values: ASK_WEATHER=5.786, ASK_MOOD=5.756, ASK_BOTH=5.997]
  π(W       ) = ASK_MOOD     [Q-values: ASK_MOOD=6.593]
  π(M       ) = ASK_WEATHER  [Q-values: ASK_WEATHER=6.593]
  π(WM      ) = RECOMMEND    [Q-values: RECOMMEND=7.500]
